<a href="https://colab.research.google.com/github/paras9o9/Suicidal-Ideation-Detection-using-Machine-Learning/blob/main/notebooks/SID_without_output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Suicidal Ideation Detection Using Machine Learing: A Comprehensive Model Analysis**

# **EDA and Data Cleaning**


In [ ]:
### EDA
import pandas as pd
import numpy as np
from collections import Counter
import re
from unicodedata import normalize

df = pd.read_csv('/content/SID_categorise_dataset.csv')

print("="*60)
print("INITIAL DATA INSPECTION")
print("="*60)
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nBasic statistics:")
print(df.describe(include='all'))

### STEP 2: CHECK MISSING VALUES
print("\n" + "="*60)
print("MISSING VALUES CHECK")
print("="*60)

missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary)

# Critical columns that shouldn't have missing values
critical_columns = ['id', 'title', 'combined_text', 'prelim_label']
for col in critical_columns:
  if col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
      print(f"\n WARNING: {missing} missing values in crtical colunm '{col}'")
      print(f"Sample rows with missing {col}:")
      print(df[df[col].isnull()].head())

### STEP 3: CHECK DUPLICATES
print("\n" + "="*60)
print("DUPLICATES DETECTION")
print("="*60)

# Check exact duplicate rows
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicates}")

# Check duplicate post IDs (if post_id colunm exists)
if 'id' in df.columns:
  duplicate_post_ids = df['id'].duplicated().sum()
  print(f"Duplicate ids: {duplicate_post_ids}")
  if duplicate_post_ids > 0:
    print("Sample duplicate post_ids:")
    dup_ids = df[df['id'].duplicated(keep=False)].sort_values('id')
    print(dup_ids[['id', 'combined_text']].head(10))

# Check duplicate text
if 'combined_text' in df.columns:
  duplicate_texts = df['combined_text'].duplicated().sum()
  print(f"\nDuplicate text content: {duplicate_texts}")
  if duplicate_texts > 0:
    print("Sample duplicate texts:")
    dup_texts = df[df['combined_text'].duplicated(keep=False)].sort_values('combined_text')
    print(dup_texts[['id', 'combined_text']].head(10))

print("\nChecking for near-duplicates (same user posting similar content)...")
user_post_counts = df['id'].value_counts()
prolific_users = user_post_counts[user_post_counts > 5].head(10)
print(f"Top prolific users (might have near-duplicates):")
print(prolific_users)

### STEP 4: LABEL CONSISTENCY CHECK
print("\n" + "="*60)
print("LABEL CONSISTENCY")
print("="*60)

if 'prelim_label' in df.columns:
  print("Unique labels found:")
  print(df['prelim_label'].value_counts())

  unique_labels_raw = df['prelim_label'].unique()
  print(f"\nRaw unique labels (check for case/whitespace issues):")
  print([f"'{prelim_label}'" for prelim_label in unique_labels_raw])

  null_labels = df['prelim_label'].isnull().sum()
  if null_labels > 0:
    print(f"\n WARNING: {null_labels} posts with null labels")

### STEP 5: TEXT QUALITY CHECKS
print("\n" + "="*60)
print("TEXT QUALITY CHECKS")
print("="*60)

if 'combined_text' in df.columns:
  df['combined_text_length'] = df['combined_text'].fillna('').astype(str).str.len()

  print(f"Text legnth statistics:")
  print(df['combined_text_length'].describe())

  empty_texts = (df['combined_text'].isnull() | (df['combined_text'].str.strip() == '')).sum()
  very_short = (df['combined_text_length'] < 10).sum()
  very_long = (df['combined_text_length'] > 1000).sum()

  print(f"\nEmpty/null texts: {empty_texts}")
  print(f"Very short texts (<10 chars): {very_short}")
  print(f"Very long texts (>1000 chars): {very_long}")

  if very_short > 0:
    print("\nSample very short texts:")
    print(df[df['combined_text_length'] < 10][['id', 'combined_text', 'prelim_label']].head())

  # Check for common artifacts
  print("\nChecking for common text artifacts...")
  if '[deleted]' in df['combined_text'].values or '[removed]' in df['combined_text'].values:
    deleted_count = df['combined_text'].str.contains(r'\[deleted\]|\[removed\]', na=False).sum()
    print(f"Posts with [deleted] or [removed]: {deleted_count}")


  ### STEP 6: USER ID VALIDATION
  print("\n" + "="*60)
  print("USER ID VALIDATION")
  print("="*60)

  if 'id' in df.columns:
    print(f"Total unique users: {df['id'].nunique()}")
    print(f"Total posts: {len(df)}")
    print(f"Average posts per user: {len(df) / df['id'].nunique():.2f}")

    posts_per_user = df['id'].value_counts()
    print(f"\nPosts per user distribution:")
    print(f" Min: {posts_per_user.min()}")
    print(f" Median: {posts_per_user.median()}")
    print(f" Mean: {posts_per_user.mean()}")
    print(f" Max: {posts_per_user.max()}")

    prolific_threshold = posts_per_user.quantile(0.95)
    prolific_users = posts_per_user[posts_per_user > prolific_threshold]
    print(f"\nUsers with >95th percentile posts (>{prolific_threshold:.0f} posts): {len(prolific_users)}")
    print("Top 10 most prolific users:")
    print(prolific_users.head(10))

### STEP 7: CROSS-VALIDATION CHECKS
print("\n" + "="*60)
print("CROSS-VALIDATION CHECKS")
print("="*60)

if 'id' in df.columns and 'prelim_label' in df.columns:
  print("Checking user label patterns (for user-level stratification)...")

  # Find users with mixed labels
  user_label_diversity = df.groupby('id')['prelim_label'].nunique()
  mixed_label_users = (user_label_diversity > 1).sum()
  single_label_users = (user_label_diversity == 1).sum()

  print(f"\nUsers with single label type: {single_label_users}")
  print(f"Users with mixed labels: {mixed_label_users}")

  if mixed_label_users > 0:
    print("\nSample users with mixed labels:")
    mixed_users = user_label_diversity[user_label_diversity > 1].index[:5]
    for user in mixed_users:
      user_posts = df[df['id'] == user]
      print(f"\n User {user}: {user_posts['prelim_label'].value_counts().to_dict()}")

### STEP 8: GENERATE CLEANING RECOMMENDATIONS
print("\n" + "="*60)
print("CLEANING RECOMMENDATIONS")
print("="*60)

recommendations = []

if df.isnull().any().any():
  recommendations.append("Handle missing values (drop or impute)")

if exact_duplicates > 0:
  recommendations.append(f"Remove {exact_duplicates} exact duplicate rows")

if 'combined_text' in df.columns and duplicate_texts > 0:
  recommendations.append(f"Deduplicate {duplicate_texts} posts with indectical text")

if 'combined_text' in df.columns and empty_texts > 0:
  recommendations.append(f"Remove or investigate {empty_texts} empty/null text posts")

if 'combined_text' in df.columns and very_short > 0:
  recommendations.append(f"Review {very_short} very short posts (<10 chars) - may be low quality")

if 'prelim_label' in df.columns:
  unique_labels = df['prelim_label'].unique()
  if any(isinstance(label, str) and (label != label.strip() or label != label.upper()) for label in unique_labels):
    recommendations.append("Standardize label casting and whitespace")

if 'id' in df.columns and df['id'].isnull().any():
  recommendations.append(f"Handle {df['id'].isnull().sum()} posts with missing IDs")

print("\nRecommended cleaning steps before splitting:")
if recommendations:
  for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")
else:
  print("No major data quality issues detected!")


print("\n" + "="*60)
print("EDA Complete - Review results and proceed with cleaning")
print("="*60)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df['prelim_label'].hist(bins=20)
plt.xlabel('Value')
plt.ylabel('Frequnecy')
plt.title('Distribution of labels')
plt.show()

In [ ]:
### DATA CLEAINING

df = pd.read_csv("/content/SID_CLEANED_DATASET.csv")

# 0) Basic text normalization helpers
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return s
    s = normalize("NFKC", s)
    s = s.replace("…", "...")  # unify ellipses
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["combined_text"] = df["combined_text"].astype(str)
df["combined_text_norm"] = df["combined_text"].apply(normalize_text)

# 1) Drop missing/empty text (post- normalization)
empty_mask = df["combined_text_norm"].isna() | (df["combined_text_norm"].str.strip() == "") | (df["combined_text_length"]==0)
df_clean = df[~empty_mask].copy()

# 2) Preserve SI-like ultra-short exceptions before short-text filtering
si_short_keywords = [
    r"\bkill\b", r"\bdead\b", r"\bdie\b", r"\bend (it|my life)\b", r"\bsuicide\b",
    r"\bself[- ]?harm\b", r"\boverdose\b", r"\bhanging\b"
]
si_short_regex = re.compile("|".join(si_short_keywords), flags=re.IGNORECASE)

df_clean["is_very_short"] = df_clean["combined_text_norm"].str.len() < 10
pattern_str = "|".join(si_short_keywords)
df_clean["short_has_si_signal"] = df_clean["combined_text_norm"].str.extract(
    pattern_str, flags=re.IGNORECASE
    ).notna().any(axis=1)

# Keep very short samples only if they show SI signal or are labeled SI
keep_short = df_clean["is_very_short"] & (df_clean["short_has_si_signal"] | (df_clean["prelim_label"]=="SI"))
drop_short_mask = df_clean["is_very_short"] & (~keep_short)
df_clean = df_clean[~drop_short_mask].copy()

# 3) Deduplicate by normalized text, keeping the first occurrence
before = len(df_clean)
df_clean = df_clean.sort_values(["post_date", "score"], ascending=[True, False])
df_clean = df_clean.drop_duplicates(subset=["combined_text_norm"], keep="first").copy()
after = len(df_clean)
print(f"Removed {before - after} exact text duplicates after normalization.")

# 4) Decide handling of UNKNOWN: here we exclude from modeling splits and save separately
df_unknown = df_clean[df_clean["prelim_label"]=="UNKNOWN"].copy()
df_model = df_clean[df_clean["prelim_label"]!="UNKNOWN"].copy()

print("Final sizes:")
print("  For modeling (no UNKNOWN):", len(df_model))
print("  UNKNOWN held out:", len(df_unknown))

# Save artifacts
df_model.drop(columns=["combined_text_norm","is_very_short","short_has_si_signal"], errors="ignore").to_csv("SID_CLEANED_DATASET.csv", index=False)
df_unknown.drop(columns=["combined_text_norm","is_very_short","short_has_si_signal"], errors="ignore").to_csv("SID_unknown_pool.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df['prelim_label'].hist(bins=20)
plt.xlabel('Value')
plt.ylabel('Frequnecy')
plt.title('Distribution of labels')
plt.show()

In [ ]:
print("\n" + "="*60)
print("LABEL CONSISTENCY")
print("="*60)

if 'prelim_label' in df.columns:
  print("Unique labels found:")
  print(df['prelim_label'].value_counts())

  unique_labels_raw = df['prelim_label'].unique()
  print(f"\nRaw unique labels (check for case/whitespace issues):")
  print([f"'{prelim_label}'" for prelim_label in unique_labels_raw])

  null_labels = df['prelim_label'].isnull().sum()
  if null_labels > 0:
    print(f"\n WARNING: {null_labels} posts with null labels")

# **User-level stratified (train/validation/test)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import json

In [ ]:
df = pd.read_csv('/content/SID_CLEANED_DATASET.csv')

print(f"Total posts: {len(df)}")
print(f"Total unique users: {df['id'].nunique()}")
print(f"\nOriginal label distribution:")
print(df['prelim_label'].value_counts())

### Assinging a label to each user based on their posts
def assign_user_label(user_posts):
  labels = user_posts['prelim_label'].values
### Priority: SI > MH > NEU > HUMOR
  if 'SI' in labels:
    return 'SI'
  elif 'MH' in labels:
    return 'MH'
  elif 'NEU' in labels:
    return 'NEU'
  else:
    return 'HUMOR'

### Group by user and assign user-level labels
user_labels = df.groupby('id').apply(assign_user_label).reset_index()
user_labels.columns = ['user_id', 'user_label']

print(f"\nUser-label label distribution")
print(user_labels['user_label'].value_counts())

### First Split: train vs (val+test) = 70/30
train_users, temp_users = train_test_split(
    user_labels['user_id'],
    test_size=0.30,
    stratify=user_labels['user_label'],
    random_state=42
)

### Second split: val vs test = 15/15 (50/50 of the 30%)
temp_user_labels = user_labels[user_labels['user_id'].isin(temp_users)]
val_users, test_users = train_test_split(
    temp_user_labels['user_id'],
    test_size=0.50,
    stratify=temp_user_labels['user_label'],
    random_state=42
)

print(f"\nUser split sizes:")
print(f"Train users: {len(train_users)} ({len(train_users)/len(user_labels)*100:.1f}%)")
print(f"Val users: {len(val_users)} ({len(val_users)/len(user_labels)*100:.1f}%)")
print(f"Test users: {len(test_users)} ({len(test_users)/len(user_labels)*100:.1f}%)")

### Assinging split labels to all posts based on their user
def assign_split(user_id):
  if user_id in train_users.values:
    return 'train'
  elif user_id in val_users.values:
    return 'val'
  elif user_id in test_users.values:
    return 'test'
  else:
    return 'unknown'

df['split'] = df['id'].apply(assign_split)

### Validation checks
print("\n=== VALIDATION CHECKS ===")

user_split_counts = df.groupby('id')['split'].nunique()
if (user_split_counts > 1).any():
  print("WARNING: Some users appear in multiple splits!")
  print(user_split_counts[user_split_counts > 1])
else:
  print("No user leakage: Each user appears in extactly one split")

unknown_count = (df['split'] == 'unknown').sum()
if unknown_count > 0:
  print(f"WARNING: {unknown_count} posts have unknown split assignment")
else:
  print("\n=== SPLIT STATISTICS ===")


for split_name in ['train', 'val', 'test']:
  split_df = df[df['split'] == split_name]
  split_users = split_df['id'].nunique()
  split_posts = len(split_df)

  print(f"\n{split_name.upper()} SET:")
  print(f" USERS: {split_users}")
  print(f" POSTS: {split_posts}")
  print(f" LABEL DISTRIBUTION:")
  for label, count in split_df['prelim_label'].value_counts().items():
    percentage = count / split_posts * 100
    print(f" {label}: {count} ({percentage:.1f}%)")

print("=== SI PREVALENCE CHECK ===")
for split_name in ['train', 'val', 'test']:
  split_df = df[df['split'] == split_name]
  si_count = (split_df['prelim_label'] == 'SI').sum()
  si_prevalance = si_count / len(split_df) * 100
  print(f"{split_name.upper()}: {si_count}/{len(split_df)} SI posts ({si_prevalance:.1f}%)")

output_file = 'SID_data_with_splits.csv'
df.to_csv(output_file, index=False)
print(f"\n Saved dataset with splits to: {output_file}")

user_split_mapping = df[['id', 'split']].drop_duplicates().sort_values('id')
user_split_mapping.to_csv('user_split_mapping.csv', index=False)
print(" Saved user-split mapaping to: user_split_mapping.csv")

summary_stats = {
    'total_posts': len(df),
    'total_users': df['id'].nunique(),
    'split_ratios': {
        'train': len(train_users) / len(user_labels),
        'val': len(val_users) / len(user_labels),
        'test': len(test_users) / len(user_labels)
    },
    'post_distribution': {
        split: {
            'users': int(df[df['split'] == split]['id'].nunique()),
            'posts': int(len(df[df['split'] == split])),
            'labels': df[df['split'] == split]['prelim_label'].value_counts().to_dict()
        }
        for split in ['train', 'val', 'test']
    }

}

with open('split_summary_stats.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)
print(" Saved summary statistics to: split_summary_stats.json")

print("=== SPLITTING COMPLETE ===")
print("Next steps:")
print("1. Review split_summary_stats.json to verify class balance")
print("2. Commit user_split_mapping.csv and split_summary_stats.json to GitHub")
print("3. Proceed with pilot annotation on training set samples")

# **Pilot Annotation**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report

In [ ]:
# Config
file_a = "/content/pilot_ann_annotatoer_A - Sheet1.csv"
file_b = "/content/pilot_ann_annotatoer_B - Sheet1.csv"

COL_ID = "id"
COL_TEXT = "combined_text"
COL_LABEL = "prelim_label"
COL_UNCERT = "uncertainty_flag"
COL_NOTES = "rationale_notes"

allowed_labels = ["SI", "MH", "NEU", "HUMOR"]

# Loading and normalising
def load_file(path):
    df = pd.read_csv(path)
    cols_lower = {c.lower().strip(): c for c in df.columns}

    col_map = {}
    mapping_candidates = {
        COL_ID: ["id", "post_id"],
        COL_TEXT: ["combined_text", "text", "body"],
        COL_LABEL: ["label_final", "final_label", "label", "prelim_label"],
        COL_UNCERT: ["uncertainty_flag", "uncertain", "uncertainty", "flag_uncertainty"],
        COL_NOTES: ["rationale_notes", "notes", "rationale", "comment"],
    }

    for std_name, potential_names in mapping_candidates.items():
        for cand in potential_names:
            if cand in df.columns:
                col_map[std_name] = cand
                break
            lc = cand.lower().strip()
            if lc in cols_lower:
                col_map[std_name] = cols_lower[lc]
                break

    required = [COL_ID, COL_TEXT, COL_LABEL]
    for r in required:
        if r not in col_map:
            raise ValueError(
                f"Required column '{r}' not found in {path}. Looked for: {mapping_candidates.get(r)}. Found columns: {df.columns.tolist()}"
            )

    keep_cols_map = {
        col_map[COL_ID]: "id",
        col_map[COL_TEXT]: "combined_text",
        col_map[COL_LABEL]: "label",
        col_map.get(COL_UNCERT, "dummy_uncertainty_col"): "uncertainty_flag",
        col_map.get(COL_NOTES, "dummy_notes_col"): "notes",
    }
    keep_cols_map = {orig: std for orig, std in keep_cols_map.items() if orig in df.columns}
    out = df[list(keep_cols_map.keys())].rename(columns=keep_cols_map).copy()

    # Normalize labels
    out["label"] = out["label"].astype(str).str.strip().str.upper()
    out["label_clean"] = np.where(out["label"].isin(allowed_labels), out["label"], "INVALID")

    # Normalize uncertainty
    if "uncertainty_flag" in out.columns:
        out["uncertainty_flag"] = out["uncertainty_flag"].astype(str).str.strip().str.lower()
        out["uncertainty_flag"] = out["uncertainty_flag"].apply(
            lambda x: 1 if x in ["yes", "true", "1"] else (0 if x in ["no", "false", "0", "", "nan"] else x)
        )
        out["uncertainty_flag"] = pd.to_numeric(out["uncertainty_flag"], errors="coerce").fillna(0).astype(int)
    else:
        out["uncertainty_flag"] = 0

    if "notes" not in out.columns:
        out["notes"] = ""

    return out

a = load_file(file_a)
b = load_file(file_b)

# Normalize IDs
a["id"] = a["id"].astype(str).str.strip()
b["id"] = b["id"].astype(str).str.strip()
print("A rows:", len(a), "B rows:", len(b), "ID overlap:", len(set(a["id"]) & set(b["id"])))

# Rename
a = a.rename(columns={"label": "label_A", "uncertainty_flag": "uncertainty_A", "notes": "notes_A", "label_clean": "label_clean_A"})
b = b.rename(columns={"label": "label_B", "uncertainty_flag": "uncertainty_B", "notes": "notes_B", "label_clean": "label_clean_B"})

# Align
merged = pd.merge(
    a[["id", "combined_text", "label_A", "label_clean_A", "uncertainty_A", "notes_A"]],
    b[["id", "label_B", "label_clean_B", "uncertainty_B", "notes_B"]],
    on="id",
    how="inner",
)

# Invalid labels
invalid_rows = merged[(merged["label_clean_A"] == "INVALID") | (merged["label_clean_B"] == "INVALID")]
if len(invalid_rows) > 0:
    print(f"Found {len(invalid_rows)} rows with invalid labels; saving to 'pilot_invalid_rows.csv' and excluding from agreement.")
    invalid_rows.to_csv("pilot_invalid_rows.csv", index=False)
    merged_valid = merged[(merged["label_clean_A"] != "INVALID") & (merged["label_clean_B"] != "INVALID")].copy()
else:
    merged_valid = merged.copy()

label_to_int = {c: i for i, c in enumerate(allowed_labels)}
yA_valid = merged_valid["label_clean_A"].map(label_to_int)
yB_valid = merged_valid["label_clean_B"].map(label_to_int)

# Agreement
if len(merged_valid) > 0:
    kappa = cohen_kappa_score(yA_valid, yB_valid)
    cm = confusion_matrix(yA_valid, yB_valid, labels=list(range(len(allowed_labels))))
    cm_df = pd.DataFrame(cm, index=[f"A_{c}" for c in allowed_labels], columns=[f"B_{c}" for c in allowed_labels])
    print(f"Cohen's kappa ({len(allowed_labels)}-class) for {len(merged_valid)} valid items: {kappa:.3f}")
    print("\nConfusion matrix (A rows vs B cols) for valid items:")
    print(cm_df)

    # Class-wise report with zero_division to silence warnings
    report = classification_report(
        merged_valid["label_clean_A"],
        merged_valid["label_clean_B"],
        labels=allowed_labels,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv("pilot_agreement_classwise_report.csv")
    print("\nClass-wise agreement report saved to pilot_agreement_classwise_report.csv")
else:
    print("\nNo valid items to calculate agreement.")
    kappa = np.nan
    cm_df = pd.DataFrame(columns=[f"B_{c}" for c in allowed_labels], index=[f"A_{c}" for c in allowed_labels])
    report_df = pd.DataFrame()

# Disagreements and adjudication
# Normalize raw labels for SI check to be robust
merged["label_A_norm"] = merged["label_A"].astype(str).str.strip().str.upper()
merged["label_B_norm"] = merged["label_B"].astype(str).str.strip().str.upper()

disagree = merged[merged["label_clean_A"] != merged["label_clean_B"]].copy()
print(f"\nDisagreements: {len(disagree)} of {len(merged)} items ({(len(disagree)/len(merged)*100 if len(merged)>0 else 0):.1f}%)")

# Safe optional columns
if "uncertainty_A" not in disagree.columns:
    disagree["uncertainty_A"] = 0
if "uncertainty_B" not in disagree.columns:
    disagree["uncertainty_B"] = 0

disagree["has_uncertainty"] = (disagree["uncertainty_A"].astype(int) > 0) | (disagree["uncertainty_B"].astype(int) > 0)
disagree["involves_SI"] = (disagree["label_A_norm"] == "SI") | (disagree["label_B_norm"] == "SI")

disagree["combined_text"] = disagree["combined_text"].astype(str)
disagree["text_len"] = disagree["combined_text"].str.len()
disagree = disagree.sort_values(by=["involves_SI", "has_uncertainty", "text_len"], ascending=[False, False, False])

adjud_cols = [
    "id",
    "combined_text",
    "label_A",
    "label_B",
    "uncertainty_A",
    "uncertainty_B",
    "notes_A",
    "notes_B",
    "involves_SI",
    "has_uncertainty",
    "text_len",
]
adjud_cols_present = [col for col in adjud_cols if col in disagree.columns]
disagree_out = disagree[adjud_cols_present].copy()
disagree_out["final_label"] = ""
disagree_out["adjudicator_notes"] = ""
disagree_out.to_csv("pilot_disagreements_for_adjudication.csv", index=False)
print("\nDisagreements for adjudication saved to pilot_disagreements_for_adjudication.csv")

all_items_adjud_cols = [
    "id",
    "combined_text",
    "label_A",
    "label_B",
    "label_clean_A",
    "label_clean_B",
    "uncertainty_A",
    "uncertainty_B",
    "notes_A",
    "notes_B",
]
all_items_present = [col for col in all_items_adjud_cols if col in merged.columns]
all_items_out = merged[all_items_present].copy()
all_items_out["final_label"] = ""
all_items_out["adjudicator_notes"] = ""
all_items_out.to_csv("pilot_all_items_adjudication_template.csv", index=False)
print("Template for all items saved to pilot_all_items_adjudication_template.csv")

# Summary CSV
summary = {
    "n_total_items_merged": int(len(merged)),
    "n_invalid_label_items_excluded_from_agreement": int(len(invalid_rows)),
    "n_valid_items_for_agreement": int(len(merged_valid)),
    "n_disagreements_among_all_merged": int((merged["label_clean_A"] != merged["label_clean_B"]).sum()),
    "disagreement_rate_among_all_merged": float(((merged["label_clean_A"] != merged["label_clean_B"]).mean() if len(merged) > 0 else 0.0)),
    "kappa_on_valid_items": float(kappa) if not np.isnan(kappa) else "N/A",
}
pd.DataFrame([summary]).to_csv("pilot_agreement_summary.csv", index=False)
cm_df.to_csv("pilot_confusion_matrix.csv", index=True)

print("\nArtifacts written:")
print("  - pilot_agreement_summary.csv")
print("  - pilot_confusion_matrix.csv")
print("  - pilot_agreement_classwise_report.csv")
print("  - pilot_disagreements_for_adjudication.csv")
print("  - pilot_all_items_adjudication_template.csv")
if len(invalid_rows) > 0:
    print("  - pilot_invalid_rows.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/SID_DATA_WITH_SPLITS.csv")

print(f"Total posts: {len(df)}")
print(f"\nLabel distribution:")
print(df['prelim_label'].value_counts())
print(f"\nSplit distribution:")
print(df['split'].value_counts())

print(f"\nMissing labels: {df['prelim_label'].isna().sum()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df['prelim_label'].hist(bins=20)
plt.xlabel('Label')
plt.ylabel('Count')
plt.title('Distribution of Labels')
plt.show()

df['split'].hist(bins=20)
plt.xlabel('Split')
plt.ylabel('Count')
plt.title('Distribution of Splits')
plt.show()

 # **Data Preprocessing**

In [ ]:
import pandas as pd
import re

df = pd.read_csv("/content/SID_DATA_WITH_SPLITS.csv")

def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'http\S+', '', text) # Remove URLs
  text = re.sub(r'[^a-z\s]', '', text) # Keeps ony letters
  text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
  return text

df['text_clean'] = df['combined_text'].apply(clean_text)

train = df[df['split'] == 'train'].copy()
val = df[df['split'] == 'val'].copy()
test = df[df['split'] == 'test'].copy()

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

# **Model Development**
# TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Calculating Term Frequnecy - Inverse Document Frequency scores.
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)) # Vectorization
X_train = vectorizer.fit_transform(train['text_clean']) # learns the vocabulary from training data transform it into a sparse matrix
X_val = vectorizer.transform(val['text_clean']) # applying the learned vocabulary
X_test = vectorizer.transform(test['text_clean']) # applying the learned vocabulary

y_train = train['prelim_label']
y_val = val['prelim_label']
y_test = test['prelim_label']

# Model Training with Class Imbalance Handling using Logistic Regression
model = LogisticRegression(
    max_iter=1000, # sufficient iteration fro convergence
    class_weight='balanced', # Handles class imbalance
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate on validation set
y_pred = model.predict(X_val)
print("=== Validation Results ===")
print(classification_report(y_val, y_pred, digits=3))

# confusion matrix
cm = confusion_matrix(y_val, y_pred, labels=['SI', 'MH', 'NEU', 'HUMOR'])
print(f"\nConfusion Matrix:")
print(pd.DataFrame(cm,
                   index=['True_SI', 'True_MH', 'True_NEU', 'True_HUMOR'],
                   columns=['Pred_SI', 'Pred_MH', 'Pred_NEU', 'Pred_HUMOR']))

# Random Froeset Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, # trains 200 trees
    max_depth=20, # limits how deep each tree can grow
    class_weight='balanced',
    random_state=42,
    n_jobs=-1 # uses all CPU cores available to speed up training/prediction
)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred_rf = rf_model.predict(X_val)
print("=== Random Forest Results ===")
print(classification_report(y_val, y_pred_rf, digits=3))

# LR v2 + SI Features

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pandas as pd

# The Expert Dictionary.
SI_KEYWORDS = {
        # Direct/Active SI expressions
        'direct': [
            'kill myself', 'end my life', 'suicidal', 'suicide', 'take my own life',
            'end it all', 'attempted suicide', 'plan to die', 'commit suicide',
            'killing myself', 'hang myself', 'shoot myself', 'overdose'
        ],

        # Passive SI expressions (death wishes without active plan)
        'passive': [
            'don\'t want to live', 'no reason to live', 'better off dead',
            'wish i was dead', 'wish i were dead', 'hope i don\'t wake up',
            'don\'t want to exist', 'don\'t want to be here', 'no longer here',
            'cease to exist', 'sleep forever', 'not wake up', 'leave this world',
            'want to disappear', 'just disappear', 'don\'t see the point',
            'no point in living', 'life has no meaning', 'no purpose',
            'tired of living', 'exhausted from living', 'can\'t keep living'
        ],

        # Indirect/coded expressions
        'indirect': [
            'unalive', 'final message', 'goodbye note', 'last post',
            'this is it', 'end tonight', 'won\'t be here tomorrow',
            'just want to be done', 'want it all to stop', 'can\'t do this anymore',
            'can\'t keep doing this', 'give up on life', 'checked out',
            'want to go home', 'ready to go', 'time to go'
        ],

        # Worthlessness and burden themes (strong SI indicators)
        'burden': [
            'life isn\'t worth it', 'better off without me', 'burden to everyone',
            'nobody needs me', 'waste of space', 'ruin everything', 'world without me',
            'everyone would be better', 'shouldn\'t exist', 'mistake to be born'
        ],

        # Preparation/planning behaviors
        'preparation': [
            'planned everything', 'set date', 'can\'t be stopped',
            'giving away things', 'delete account', 'last day',
            'made arrangements', 'got everything ready', 'writing goodbye'
        ],

        # Regional language phrases
        'hindi': [
            'chhod dena hai sab', 'khatam karna hai', 'ab aur nahi',
            'mar jaunga', 'mar jaungi', 'jaane ka time aa gaya',
            'jee nahi sakta', 'zinda nahi rehna'
        ]
    }

# Feature Extraction Function

# This function iterates through every post and generates numerical features based on your dictionary.
def extract_si_features(text):
  text_lower = str(text).lower()
  features= {}

  # It counts how many times words from each category appear.
  total_matches = 0
  for category, keyword_list in SI_KEYWORDS.items():
    category_matches = 0
    for keyword in keyword_list:
      if keyword in text_lower:
        category_matches += 1
        total_matches += 1

    features[f'{category}_count'] = category_matches
    features[f'has_{category}'] = int(category_matches > 0)

  # Aggregated features
  features['total_si_keywords'] = total_matches
  features['si_category_count'] = sum(1 for cat in SI_KEYWORDS
                                      if features.get(f'has_{cat}', 0) > 0)

  # Additional signals
  features['has_death_words'] = int(any(w in text_lower for w in
                                        ['die', 'death', 'dead', 'dying']))
  features['has_self_harm'] = int(any(w in text_lower for w in
                                      ['cut', 'harm', 'hurt myself', 'self harm']))
  # High-risk flags
  features['has_method'] = int(any(m in text_lower for m in
                                   ['hang', 'shoot', 'overdose', 'jump', 'pills']))
  features['has_planning'] = features['preparation_count']

  return features

# Apply to train/val sets
train['si_features'] = train['combined_text'].apply(extract_si_features)
val['si_features'] = val['combined_text'].apply(extract_si_features)

si_train = pd.DataFrame(train['si_features'].tolist())
si_val = pd.DataFrame(val['si_features'].tolist())

from scipy.sparse import hstack

# Combine TF-IDF + SI features

# The model now sees a combined dataset with 5020 columns. It can learn from both the genral vocabulary (TF-IDF) and your specific high-risk indicators simultaneously.
X_train_tfidf = vectorizer.fit_transform(train['text_clean']) # Is a sparse matrix with ~5000 columns.
X_train_combined = hstack([X_train_tfidf, si_train.values]) # Horizontal stack glues these two matrics together side-by-side.

X_val_tfidf = vectorizer.transform(val['text_clean'])
X_val_combined = hstack([X_val_tfidf, si_val.values])

# Train model
model_v2 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_v2.fit(X_train_combined, train['prelim_label'])

# Evaluate
y_pred_v2 = model_v2.predict(X_val_combined)
from sklearn.metrics import classification_report
print("=== LR v2 with SI Features ===")
print(classification_report(val['prelim_label'], y_pred_v2, digits=3))

# **Feature Importance Analysis**

In [ ]:
import numpy as np
tfidf_features = vectorizer.get_feature_names_out()
si_feature_names = list(si_train.columns)

si_class_idx = list(model_v2.classes_).index('SI')
coefficients = model_v2.coef_[si_class_idx]

si_feature_start = len(tfidf_features)
si_coefs = coefficients[si_feature_start:]

si_importance = pd.DataFrame({
    'Feature': si_feature_names,
    'Coefficient': si_coefs
}).sort_values('Coefficient', ascending=False)

print("\n === Top 10 SI-preditive features ===")
print(si_importance.head(10))

print("\n === Bottom 10 SI-predictive features ===")
print(si_importance.tail(10))

# Save for your thesis
si_importance.to_csv('si_features_importance.csv', index=False)

# LR v3 with Optimesed Features

In [ ]:
# Reduced noise keyword list.
SI_KEYWORDS_V2 = {
    'direct': [
        'kill myself', 'end my life', 'suicidal', 'suicide', 'take my own life',
        'end it all', 'attempted suicide', 'plan to die', 'commit suicide',
        'killing myself', 'hang myself', 'shoot myself', 'overdose'
    ],
    'indirect': [
        'unalive', 'final message', 'goodbye note', 'last post',
        'this is it', 'end tonight', 'won\'t be here tomorrow',
        'just want to be done', 'want it all to stop', 'can\'t do this anymore',
        'can\'t keep doing this', 'give up on life', 'checked out',
        'want to go home', 'ready to go', 'time to go'
    ],
    'preparation': [
        'planned everything', 'set date', 'can\'t be stopped',
        'giving away things', 'delete account', 'last day',
        'made arrangements', 'got everything ready', 'writing goodbye'
    ]
}

# Feature Extraction Function

# This function iterates through every post and generates numerical features based on your dictionary.
def extract_si_features_v2(text):
  text_lower = str(text).lower()
  features = {}

  # It counts how many times words from each category appear.
  total_matches = 0
  for category, keyword_list in SI_KEYWORDS_V2.items():
    category_matches = 0
    for keyword in keyword_list:
      if keyword in text_lower:
        category_matches += 1
        total_matches += 1

    # Aggregated features
    features[f'{category}_count'] = category_matches
    features[f'has_{category}'] = int(category_matches > 0)

  features['total_si_keywords'] = total_matches
  features['si_category_count'] = sum(1 for cat in SI_KEYWORDS_V2 if features.get(f'has_{cat}', 0) > 0)
  features['has_death_words'] = int(any(w in text_lower for w in ['die', 'death', 'dead', 'dying']))
  features['has_method'] = int(any(m in text_lower for m in ['hang', 'shoot', 'overdose', 'jump', 'pills']))

  return features

# Re-extract features
train['si_features_v3'] = train['combined_text'].apply(extract_si_features_v2)
val['si_features_v3'] = val['combined_text'].apply(extract_si_features_v2)

si_train_v3 = pd.DataFrame(train['si_features_v3'].tolist())
si_val_v3 = pd.DataFrame(val['si_features_v3'].tolist())

# Recombine wtih TF-IDF
from scipy.sparse import hstack

X_train_combined_v3 = hstack([X_train_tfidf, si_train_v3.values])
X_val_combined_v3 = hstack([X_val_tfidf, si_val_v3.values])

# Train LR v3
model_v3 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_v3.fit(X_train_combined_v3, train['prelim_label'])

y_pred_v3 = model_v3.predict(X_val_combined_v3)
print("=== LR v3 with SI Features ===")
print(classification_report(val['prelim_label'], y_pred_v3, digits=3))

# LR v4 with Custom Weights

In [ ]:
# Aggressively boost SI detection

# Manual class weighting to shift the model's prority toward detecting suicicdal ideation, even at the cost of more false alarms.
custom_weights = {
    'SI': 3, # Up from 2.2 (balanced)
    'MH': 1.0,
    'NEU': 2.0, # Help small class
    'HUMOR': 1.0
}
# Why Do this?
"""By setting class weight, we explicilty tell the model:
Making a mistake on an SI case is 3 times worse than making a on Humor case."""

model_v4 = LogisticRegression(
    max_iter=1000,
    class_weight=custom_weights,
    random_state=42
)
model_v4.fit(X_train_combined_v3, train['prelim_label'])
y_pred_v4 = model_v4.predict(X_val_combined_v3)

print("=== LR v4 with Custom Class Weights ===")
print(classification_report(val['prelim_label'], y_pred_v4, digits=3))


# LinearSVM

In [ ]:
from sklearn.svm import LinearSVC

# SVM focuses strictly on finding the widest geometric gap between classes.
svm = LinearSVC(
    class_weight='balanced',
    max_iter=5000, # SVM can sometimes be harder to converge than LR, thus higher iteration limt prevents 'ConvergenceWarning' errors.
    random_state=42,
    dual=False # Rule of thumb to prefer 'dual=False', cause it solves the 'primal' problem, whcih is faster and often more stable for larger dataset.
)
svm.fit(X_train_combined_v3, train['prelim_label'])
y_pred_svm = svm.predict(X_val_combined_v3)

print("=== LinearSVC Results ===")
print(classification_report(val['prelim_label'], y_pred_svm, digits=3))

# Fine-tune Weights

In [ ]:
# This code snippet performs a hyperparameter grid search specificaly for the class weights, which is crucial step in fine-tuning the model's sensitivity to suicidal ideation.

# List of increasing weight
weights_to_test = [3.0, 3.5, 4.0, 4.5, 5.0]
results = []

# For every weight 'w', it builds a fresh LR model where the penality for missing an 'SI' case is 'w' times higher than normal.
for w in weights_to_test:
  custom_weights = {'SI': w, 'MH': 1.0, 'NEU': 2.0, 'HUMOR': 1.0}

  model = LogisticRegression(max_iter=1000, class_weight=custom_weights, random_state=42)
  model.fit(X_train_combined_v3, train['prelim_label'])
  y_pred = model.predict(X_val_combined_v3)

  # It specifically extracts Precision, Recall, and F1-Score for the 'SI' class only.
  from sklearn.metrics import precision_recall_fscore_support
  precision, recall, f1, _ = precision_recall_fscore_support(val['prelim_label'], y_pred, labels=['SI'], average=None)

  results.append({
      'SI_Weight': w,
      'SI_Recall': recall[0],
      'SI_Precision': precision[0],
      'SI_F1': f1[0]
  })

results_df = pd.DataFrame(results)
print(results_df)
# Classic machine learning phenomenon called the Precision-Rcall Trade-off.

# **Deep Learning**
# **Phase-1**: Multi-Class DistilBERT (SI vs MH vs Humour vs Neutral)
A Transformr model. Think of it as a pre-trained brain that has already read all of Wikipedia. It understands English grammer, slang, and context. "Distil" means it's a smaller, faster version of the gaint BERT model. "Uncased" means it treats "Help" and "help" as the same word.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

# Configuration
FILE_PATH = "/content/SID_DATA_WITH_SPLITS.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
SEED = 42

print("Loading data...")
df = pd.read_csv(FILE_PATH)

# Checking for the 'text' and 'label' columns.
if "combined_text" not in df.columns or "prelim_label" not in df.columns:
  raise ValueError("CSV must have 'combined_text' and 'prelim_label' columns")

# Map labels to integers, ensuring consistent order.
unique_labels_sorted = sorted(df['prelim_label'].unique())
label_map = {label: i for i, label in enumerate(unique_labels_sorted)} # Creates a consistent numeric ID for each class, and the model is trained to predict these integers (not the strings).
df['labels'] = df['prelim_label'].map(label_map)
num_labels = len(unique_labels_sorted)

"""THe 'tain_test_split' calls ensure each split keeps approxmiately the same class proportion,
whcih is important when SI is rare (otherwise our val/test might) accidentally contain too few SI examples."""
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['labels'], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['labels'], random_state=SEED)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # Loads the DistilBERT tokenizer rules for turning text into token IDs that match the model's vocabulary.

def tokenize_function(examples):
  """Makes every example exactly length 128 by cutting long posts and padding short post,
   producing: input_ids (token numbers the model reads) and attention_mask (1 for real token, 0 for padding)."""
  return tokenizer(examples["combined_text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Setting format to include the new 'labels' column.
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Calculate class weights for multi-class classification.
# Ensure class_counts are based on the new integer 'labels' column for consistency.
class_counts_numeric = train_df['labels'].value_counts().sort_index()
total_samples = sum(class_counts_numeric)
# Inverse frequency weights
class_weights = torch.tensor([total_samples / count for count in class_counts_numeric.values], dtype=torch.float)

print(f"Using Class Weights: {class_weights}")

# The trainer uses the model's standardloss, but we override 'compute_loss' to use 'CrossEntropyLoss' so minority classes contribute more to the loss.

"""In PyTorch, 'CrossENtropyLoss' accepts a per-class 'weight' tensor that scales the loss for each class index, higher weight means mistakes on that
class are penalized more strongly. That's exactly how our code tries to reduce 'model ignores SI becuase it's rare' behavior, which is a common reason
people customize Trainer loss for imblanced datasets."""

"""Our trainer, getting a 'Safe' post wrong costs 1 point. Getting a 'Suicidal' post wrong costs 13.5 points (from our weight calulation). The model
tries desperalty to avoid getting Suicide posts wrong becouse it 'hurts' its score so much."""

class WeightedTrainer(Trainer):
  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    labels = inputs.get("labels")
    outputs = model(**inputs)
    logits = outputs.get("logits")
    loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
    loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
    return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels) # Loads DistilBERT plus a small classification layer that outputs 'num_labels' logits per text.
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

"""Hugging Face Trainer handls the training loop (batches, optimizer steps, batch size, evaluation each epoch, checkpointing)
using our TrainingArguments settings like epochs, batch size, evaluation startegy, and "load best model at end"."""
training_args = TrainingArguments(
    output_dir = "./results",
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    num_train_epochs = EPOCHS,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir='./logs',
)

# Collectiong data together.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    data_collator = data_collator,
)

print("Starting training...")
trainer.train()

print("Generating predictions...")
predictions = trainer.predict(tokenized_test) # Return logits for each test item.
logits = predictions.predictions
# For multi-class, we typically take the argmax of the logits for predictions.
y_pred_labels = np.argmax(logits, axis=1) # Converts "score per class" into one predicted class index per item.
y_true = predictions.label_ids

# Use the original label names for the classification report.
target_names = [label for label, i in sorted(label_map.items(), key=lambda item: item[1])]

print("\n--- FINAL THESIS RESULTS (DistilBERT) ---")
print(classification_report(y_true, y_pred_labels, target_names=target_names)) # Prints precision/recall/F1 per class, and 'confusion_marix shows which classes get confused with which.
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_labels))

# Removed threshold optimization block as it's for binary classification.

# Binary classification Pipeline

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

FILE_PATH = "/content/SID_DATA_WITH_SPLITS.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
SEED = 42

print("Loading data...")
df = pd.read_csv(FILE_PATH)

if "combined_text" not in df.columns or "prelim_label" not in df.columns:
    raise ValueError("CSV must have 'combined_text' and 'prelim_label' columns")

# BINARY MAPPING IN PANDAS

# We do this BEFORE creating Datasets to avoid map/key errors.
print("Original Label Counts:")
print(df['prelim_label'].value_counts())

# Assuming 'Suicidal Ideation' is the string label for SI.
si_label_str = 'SI'

if si_label_str not in df['prelim_label'].unique():
     # Fallback: assume the class with fewer samples is SI (usually true for imbalanced data)
     print(f"Warning: '{si_label_str}' not found. Using minority class as SI.")
     si_label_str = df['prelim_label'].value_counts().idxmin()

print(f"Mapping '{si_label_str}' to 1 (SI), everything else to 0.")

# THe explicit mapping.

"""By grouping MHm Neutral, and Humor into a single "Non-Suicidal class (0), you force the model to
learn only the specific features that distinguish active suicide risk from everything else."""
def binary_mapper(val):
    return 1 if val == si_label_str else 0

df['binary_label'] = df['prelim_label'].apply(binary_mapper)

print("New Binary Label Distribution:")
print(df['binary_label'].value_counts())

# Stratified on New Binary Label.

"""It gurarantees that if 10% of our data is suicidal, exactly 10% of our Train set and 10% of
your Test set will be suicidal. It keeps the 'difficulty' fair across all splits."""
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['binary_label'], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['binary_label'], random_state=SEED)

# CREATE DATASETS

# We rename 'binary_label' to 'labels' because Hugging Face Trainer expects 'labels' key.
train_dataset = Dataset.from_pandas(train_df[['combined_text', 'binary_label']].rename(columns={'binary_label': 'labels'}))
val_dataset = Dataset.from_pandas(val_df[['combined_text', 'binary_label']].rename(columns={'binary_label': 'labels'}))
test_dataset = Dataset.from_pandas(test_df[['combined_text', 'binary_label']].rename(columns={'binary_label': 'labels'}))

# CALCULATE WEIGHTS

"""Here we are telling the model, 'Pay attention to the minority class(SI).' The 1.5 mutilplier is an agressive hyperparameter.
It means we are upweighting the risk class even more than the mathematical blance suggests."""
counts = train_df['binary_label'].value_counts().sort_index()
neg_count = counts[0]
pos_count = counts[1]
pos_weight = (neg_count / pos_count) * 1.5
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)
print(f"Binary Class Weights: {class_weights}")

# TOKENIZATION & MODEL

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["combined_text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("Tokenizing data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Setting format
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Clear cache
if 'model' in locals(): del model
if torch.cuda.is_available(): torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='./logs',
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Starting Binary Training...")
trainer.train()

# **Post-Hoc Optimization**

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score

# GENERATING RAW PROBILITIES

print("\n--- GENERATING FINAL THESIS METRICS ---")
predictions = trainer.predict(tokenized_test)
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)[:, 1].numpy() # Extract the raw confidence score.
y_true = predictions.label_ids

# SCAN FOR THE "SWEET SPOT"

print(f"{'Threshold':<10} {'Recall (SI)':<12} {'Precision':<10} {'F1 Score':<10}")
print("-" * 45)

best_f1 = 0
best_thresh = 0.5
final_metrics = {}

# This loop treats evry possible "Sensitivity setting" from 10% to 95%.
for t in np.arange(0.1, 0.95, 0.05):
    y_pred = (probs >= t).astype(int)
    rec = recall_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred)

    # Print row
    print(f"{t:.2f}       {rec:.4f}       {prec:.4f}     {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        final_metrics = {'rec': rec, 'prec': prec, 'f1': f1}

print("-" * 45)
print(f"🏆 BEST F1 THRESHOLD: {best_thresh:.2f}")
print(f"   Recall:    {final_metrics['rec']:.2%}")
print(f"   Precision: {final_metrics['prec']:.2%}")
print(f"   F1 Score:  {final_metrics['f1']:.4f}")

# FINAL DETAILED REPORT
final_preds = (probs >= best_thresh).astype(int)
print("\n--- CLASSIFICATION REPORT (At Best Threshold) ---")
print(classification_report(y_true, final_preds, target_names=['Safe/Other', 'Suicidal']))

print("--- CONFUSION MATRIX ---")
cm = confusion_matrix(y_true, final_preds)
print(cm)
print(f"\nStats for Thesis:")
print(f"Total SI Cases in Test: {cm[1,0] + cm[1,1]}")
print(f"Caught by Model:        {cm[1,1]} (True Positives)")
print(f"Missed by Model:        {cm[1,0]} (False Negatives)")

# **Heatmap**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Safe', 'Suicidal'], yticklabels=['Safe', 'Suicidal'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title(f'Confusion Matrix (Threshold = {best_thresh:.2f})')
plt.show()

# **Error Analysis**

In [ ]:
test_df_analysis = test_df.copy().reset_index(drop=True)
test_df_analysis['prob_SI'] = probs
test_df_analysis['pred'] = final_preds
test_df_analysis['actual'] = y_true

# It creates a subset of data (fasle_negative) containing only the case where: Actual label = 1, Predicted Label = 0.
false_negatives = test_df_analysis[(test_df_analysis['actual'] == 1) & (test_df_analysis['pred'] == 0)]

# It prints the model's probabilty scroe for these errors.
print("--- EXAMPLES OF MISSED SI (False Negative) ---")
for i, row in false_negatives.head(5).iterrows():
  print(f"\nText: {row['combined_text']}")
  print(f"MOdel probabilty: {row['prob_SI']:.4f} (Threshold was {best_thresh})")

# **Local Interpretable Model-agnostic Explanations (LIME) for Explainability**

It's designed to reveal *why* our model classified a specific text as "Safe" or "Suicidal" by highlighting the most influential words.

In [ ]:
!pip install lime

import numpy as np
import torch
import torch.nn.functional as F
from lime.lime_text import LimeTextExplainer

# Tells LIME that output 0 corresponds to "Safe" and output 1 corresponds to "Suicidal", making the final visualization.
explainer = LimeTextExplainer(class_names=["Safe", "Suicidal"])

# LIME needs a function that takes raw text and returns probabilites.
def predictor(texts):
  inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)

  # Move to available GPU
  device = 'cuda' if torch.cuda.is_available() else 'cpu' # Ensures the model runs on the GPU if avaiable.
  model.to(device)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1) # Converts the model's raw logits into probobablities that sum to 1.

  return probs.cpu().numpy() # LIME expects standard numpy arrays, not pytorch tensors, so this converts the out back.

# "Cacoon" or "Falsely accused" text from your error analysis.
# text_to_explain = "Falsely accused of being a pedophile, and I can't live the same"
# text_to_explain = "I can’t stand being an ugly women I want to die"
# text_to_explain = "I hate that people call boys “zesty” or “gay” for sitting in this position. I am as straight as a ruler and think this is super comfortable."
# text_to_explain = "My back is killing me after that workout."
# text_to_explain = "I don't want to be around anymore. I'm done getting treated this way. This is so unfair."
# text_to_explain = "I’ve made up my mind. I wrote the letters yesterday and I finally feel a sense of peace knowing it will all be over tonight. Goodbye."
text_to_explain = "I am dying of laughter"

# text_to_explain = "I do NOT want to die"

print(f"Explaining  text: '{text_to_explain}'")
exp = explainer.explain_instance(text_to_explain, predictor, num_features=10)

exp.show_in_notebook(text=True)

exp.save_to_file("lime_explanaiton.html")
print("Saved explanation to lime_explanation.html")

# **Phase-2**: Multi-Class DistilBERT (SI vs MH vs Humour vs Neutral)

In [ ]:
!pip install lime

import numpy as np
import torch
import gc
import torch.nn.functional as F
from lime.lime_text import LimeTextExplainer

# Tells LIME that output 0 corresponds to "Safe" and output 1 corresponds to "Suicidal", making the final visualization.
explainer = LimeTextExplainer(class_names=["Safe", "Suicidal"])

# LIME needs a function that takes raw text and returns probabilites.
def predictor(texts):
    # Ensure model is in eval mode
    model.eval()

    # Disable gradient calculation to save memory
    with torch.no_grad():
        # Your tokenization and prediction logic here
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to("cuda")
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return probs.cpu().detach().numpy()

gc.collect()
torch.cuda.empty_cache

# "Cacoon" or "Falsely accused" text from your error analysis.
# text_to_explain = "Falsely accused of being a pedophile, and I can't live the same"
# text_to_explain = "I can’t stand being an ugly women I want to die"
# text_to_explain = "I hate that people call boys “zesty” or “gay” for sitting in this position. I am as straight as a ruler and think this is super comfortable."
# text_to_explain = "My back is killing me after that workout."
text_to_explain = """I don't think I can do this anymore. It’s not just a bad day or a bad week, it’s been years of feeling like a
ghost in my own life. I woke up today and realized that I genuinely don't want to see tomorrow. Everyone says it gets better, but it only
seems to get worse for me. I feel like such a burden to my parents and my friends; they would honestly be so much freer without having to worry about
me all the time. I’ve started giving away my things and cleaning up my room so they won't have to deal with a mess when I'm gone.
It feels strangely peaceful to finally have a plan to end the noise in my head. I just wanted to say this somewhere because I can't tell anyone I know."""

print(f"Explaining  text: '{text_to_explain}'")
exp = explainer.explain_instance(
    text_to_explain,
    predictor,
    num_features=10,
    num_samples=500  # Optional: Lower samples if it's still too slow (default is 5000)
)

exp.show_in_notebook(text=True)

exp.save_to_file("lime_explanaiton.html")
print("Saved explanation to lime_explanation.html")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

# Configuration
FILE_PATH = "/content/SID_DATA_WITH_SPLITS.csv"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2
SEED = 42

print("Loading data for Multi-Class Analysis...")
df = pd.read_csv(FILE_PATH)

# Multi-Class setup
print("Original Label Counts:")
print(df['prelim_label'].value_counts())

# Creating label mapping for 4 classes (alphabetical order for consistency).
label_map = {'HUMOR': 0, 'MH': 1, 'NEU': 2, 'SI': 3}
df['multi_label'] = df['prelim_label'].map(label_map)

print("\nMulti-Class Distribution:")
print(df['multi_label'].value_counts().sort_index())

# Data Split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['multi_label'], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['multi_label'], random_state=SEED)

train_dataset = Dataset.from_pandas(train_df[['combined_text', 'multi_label']].rename(columns={'multi_label': 'labels'}))
val_dataset = Dataset.from_pandas(val_df[['combined_text', 'multi_label']].rename(columns={'multi_label': 'labels'}))
test_dataset = Dataset.from_pandas(test_df[['combined_text', 'multi_label']].rename(columns={'multi_label': 'labels'}))

# Tokenization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
  return tokenizer(examples['combined_text'], padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("Tokenizing...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", 'labels'])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Multi-Class Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Standard Training (no custom weights needed for balanced multi-class).
training_args = TrainingArguments(
    output_dir = "./multi_class_results",
    learning_rate = 2e-5,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = BATCH_SIZE,
    num_train_epochs = EPOCHS,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    logging_dir = './multi_class_logs',
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    data_collator = data_collator,
)

print("Starting Mulit-Class Training...")
trainer.train()

# Evaluating
print("\nGenerating Multi-Class Prediction...")
prediction = trainer.predict(tokenized_test)
probs = torch.nn.functional.softmax(torch.tensor(prediction.predictions), dim=-1).numpy()
y_true = prediction.label_ids
y_pred = np.argmax(probs, axis=1)

class_names = ['HUMOR', 'MH', 'NEU', 'SI']

print("\n --- MULTI-CLASS CLASSIFICATION REPORTS ---")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\n -- CONFUSION MATRIX ---")
cm = confusion_matrix(y_true, y_pred)
print(cm)

# Pretty print with labels.
print("\nConfusion Matrix (Predicted \ Actual):")
header = "     " + " ".join([f"{name:>6}" for name in class_names])
print(header)
for i, row in enumerate(cm):
  row_str = f"{class_names[i]:>4} |" + " ".join([f"{val:>6}" for val in row])
  print(row_str)

# Clinical Insight: MH vs SI confusion.
mh_si_ocnfusion = cm[1:3, 1:3]
print(f"\n MH vs SI Confusion (rows=Actual, cols=Predicted):")
print("         Pred MH  Pred SI")
print(f"Actual MH | {cm[1, 1]:>5} {cm[1, 3]:>5}")
print(f"Actual SI | {cm[3, 1]:>5} {cm[3, 3]:>5}")

mi_to_si = cm[1, 3] # Mental Health predicted as Suicide (False Positives).
si_to_mh = cm[3, 1] # Suicide predicted as Mental Health (False Positives).

print(f"\nClinical Insights:")
print(f" False Alarms (MH->SI): {mi_to_si} cases ({mi_to_si/(cm[1, 1]+cm[1, 3])*100:.1f}%) ")
print(f" Missed SI (SI->MH): {si_to_mh} cases ({si_to_mh/(cm[3, 1]+cm[3, 3])*100:.1f}%) ")


In [ ]:
!pip install lime

import numpy as np
import torch
import torch.nn.functional as F
from lime.lime_text import LimeTextExplainer

# Tells LIME that output 0 corresponds to "Safe" and output 1 corresponds to "Suicidal", making the final visualization.
explainer = LimeTextExplainer(class_names=["Safe", "Suicidal"])

# LIME needs a function that takes raw text and returns probabilites.
def predictor(texts):
  inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)

  # Move to available GPU
  device = 'cuda' if torch.cuda.is_available() else 'cpu' # Ensures the model runs on the GPU if avaiable.
  model.to(device)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1) # Converts the model's raw logits into probobablities that sum to 1.

  return probs.cpu().numpy() # LIME expects standard numpy arrays, not pytorch tensors, so this converts the out back.

# "Cacoon" or "Falsely accused" text from your error analysis.
# text_to_explain = "Falsely accused of being a pedophile, and I can't live the same"
# text_to_explain = "I can’t stand being an ugly women I want to die"
# text_to_explain = "I hate that people call boys “zesty” or “gay” for sitting in this position. I am as straight as a ruler and think this is super comfortable."
# text_to_explain = "My back is killing me after that workout."
# text_to_explain = "I don't want to be around anymore. I'm done getting treated this way. This is so unfair."
# text_to_explain = "I’ve made up my mind. I wrote the letters yesterday and I finally feel a sense of peace knowing it will all be over tonight. Goodbye."
text_to_explain = "I am dying of laughter"

# text_to_explain = "I do NOT want to die"

print(f"Explaining  text: '{text_to_explain}'")
exp = explainer.explain_instance(text_to_explain, predictor, num_features=10)

exp.show_in_notebook(text=True)

exp.save_to_file("lime_explanaiton.html")
print("Saved explanation to lime_explanation.html")